# MagicBox training on TPU
Run each stage directly in the notebook kernel. Set `RUN_MODE` to `smoke` for 10 updates or `full` for the configured epochs. Source is loaded from the pinned GitHub checkout.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

RUN_MODE = 'smoke'  # 'smoke' or 'full'.
DEVICES = None  # Detect all visible TPUs; set 1 or 8 to require that exact count.
REQUESTS_PER_DEVICE = 1
SMOKE_STEPS = 10
EPOCHS = 3
SESSION_HOURS = 1 if RUN_MODE == 'smoke' else 8
MICROBATCHES = 1 if RUN_MODE == 'smoke' else 4
ROW_CHUNK = 1 if RUN_MODE == 'smoke' else 4
SOURCE_TOKENS = 1024
SCHEMA_TOKENS = 512
SCHEMA_ROWS = 256
SCHEMA_SEQUENCES = None  # Packed SCHEMA_TOKENS encoder rows per request; None measures the dataset.
LEARNING_RATE = 2e-5
VALIDATION_RECORDS = 8 if RUN_MODE == 'smoke' else 256
FINAL_RECORDS = 8 if RUN_MODE == 'smoke' else 0

IS_KAGGLE = Path('/kaggle/working').is_dir()
HOST = Path('/kaggle/working') if IS_KAGGLE else Path('/content')
SCRATCH = Path('/kaggle/temp/magicbox') if IS_KAGGLE else HOST / 'magicbox-scratch'
OUTPUT = None  # Defaults to magicbox-{RUN_MODE}-{DEVICES}dev under HOST.
RESUME_FROM = None  # Previous output folder with the same hardware and batch settings.
DATASET = None  # Override with a local completed dataset directory to skip its download.
DATASET_REPO = 'protodotdesign/magicbox-v1'
DATASET_REVISION = 'f074bb549f16ea091fd8ece12e79652b8082871f'
XLA_DUMP = None  # Set a directory to save text HLO for every compiled program.

if RUN_MODE not in ('smoke', 'full'):
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
for name in (
    'REQUESTS_PER_DEVICE', 'MICROBATCHES', 'ROW_CHUNK',
    'SOURCE_TOKENS', 'SCHEMA_TOKENS', 'SCHEMA_ROWS', 'SMOKE_STEPS',
):
    if type(globals()[name]) is not int or globals()[name] < 1:
        raise ValueError(f'{name} must be a positive integer')
if SCHEMA_SEQUENCES is not None and (type(SCHEMA_SEQUENCES) is not int or SCHEMA_SEQUENCES < 1):
    raise ValueError('SCHEMA_SEQUENCES must be None or a positive integer')
SCRATCH.mkdir(parents=True, exist_ok=True)
if XLA_DUMP is not None:
    os.environ['XLA_FLAGS'] = f'--xla_dump_to={XLA_DUMP} --xla_dump_hlo_as_text'
CHECKOUT = SCRATCH / 'minifield-training'
SOURCE_REPO_URL = 'https://github.com/Minifield-Labs/minifield-training.git'
SOURCE_REVISION = 'cb44160f5b3ec138d4a65b9f88d3c8bfe0b2d377'
MODEL = SCRATCH / 'encoder'
CACHE = SCRATCH / 'arrow'


SEED = 20260927
RUN_ID = 'magicbox-lfm350m-v1'

def setup_command(*command, cwd=None):
    result = subprocess.run(command, cwd=cwd or CHECKOUT, check=True,
                            text=True, stdout=subprocess.PIPE)
    print(result.stdout, end='', flush=True)
    return result.stdout

## Fetch the pinned source


In [ ]:
if not (CHECKOUT / '.git').is_dir():
    if CHECKOUT.exists():
        raise RuntimeError(f'Expected a clean checkout path: {CHECKOUT}')
    setup_command('git', 'clone', '--no-checkout', SOURCE_REPO_URL, str(CHECKOUT),
              cwd=SCRATCH)
else:
    if setup_command('git', 'status', '--porcelain').strip():
        raise RuntimeError(f'Checkout contains local changes: {CHECKOUT}')
contains_pin = subprocess.run(
    ['git', 'cat-file', '-e', f'{SOURCE_REVISION}^{{commit}}'],
    cwd=CHECKOUT, capture_output=True, check=False).returncode == 0
if not contains_pin:
    setup_command('git', 'fetch', '--depth', '1', SOURCE_REPO_URL, SOURCE_REVISION)
setup_command('git', 'checkout', '--detach', SOURCE_REVISION)
actual_revision = setup_command('git', 'rev-parse', 'HEAD').strip()
if actual_revision != SOURCE_REVISION:
    raise RuntimeError(f'Expected {SOURCE_REVISION}, got {actual_revision}')
if not (CHECKOUT / 'examples/magicbox/train.py').is_file():
    raise RuntimeError('The pinned checkout is missing the MagicBox trainer')
print('Training source:', SOURCE_REPO_URL, actual_revision)

## Install into this kernel


In [ ]:
if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Select a Python 3.12 or 3.13 notebook runtime')
if any(name in sys.modules for name in ('jax', 'jaxlib', 'minifield_training')):
    raise RuntimeError('Restart the notebook session before installing dependencies')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'uv==0.11.30'])
requirements = SCRATCH / 'notebook-requirements.txt'
subprocess.check_call([
    sys.executable, '-m', 'uv', 'export', '--project', str(CHECKOUT),
    '--locked', '--no-dev', '--all-extras', '--no-emit-project', '--no-hashes',
    '--format', 'requirements-txt', '--output-file', str(requirements),
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(requirements), str(CHECKOUT), 'jax[tpu]==0.7.2',
])
import importlib.metadata
for package in ('numpy', 'scipy', 'pandas', 'pyarrow', 'transformers', 'tokenizers'):
    loaded = sys.modules.get(package)
    if loaded is not None and getattr(loaded, '__version__', None) != importlib.metadata.version(package):
        raise RuntimeError(f'{package} changed on disk. Restart the session, then run from the first cell.')
sys.path.insert(0, str(CHECKOUT))
print('Installed into notebook kernel:', sys.executable)

## Discover TPU devices


In [ ]:
import json
from pathlib import Path
import sys
import jax

devices = jax.devices()
if jax.__version__ != '0.7.2':
    raise RuntimeError('Expected pinned JAX 0.7.2')
if jax.process_count() != 1 or len(devices) != jax.local_device_count():
    raise RuntimeError('This notebook requires all TPU devices on one host')
if not devices or any(device.platform != 'tpu' for device in devices):
    raise RuntimeError('Select a TPU runtime')
hardware = {'devices': len(devices), 'kinds': [d.device_kind for d in devices]}

print(hardware, flush=True)
if DEVICES is None:
    DEVICES = hardware['devices']
if type(DEVICES) is not int or DEVICES < 1 or hardware['devices'] != DEVICES:
    raise ValueError(f"Requested {DEVICES} devices; runtime exposes {hardware['devices']}")
REQUESTS = DEVICES * REQUESTS_PER_DEVICE
if OUTPUT is None:
    OUTPUT = HOST / f'magicbox-{RUN_MODE}-{DEVICES}dev'
OUTPUT = Path(OUTPUT)
print('Mode:', RUN_MODE, 'Devices:', DEVICES,
      'Requests per device:', REQUESTS_PER_DEVICE,
      'Logical batch:', REQUESTS * MICROBATCHES, 'Output:', OUTPUT)

## Download dataset and encoder


In [ ]:
from huggingface_hub import snapshot_download
from minifield_training.models.lfm2_5 import encoder
import shutil

if DATASET is None:
    DATASET = SCRATCH / 'dataset'
    snapshot_download(DATASET_REPO, repo_type='dataset', revision=DATASET_REVISION,
                      local_dir=str(DATASET), allow_patterns=[
                          'manifest.json', 'identity.json', 'report.json', 'sources.json',
                          'tokenizer/*', 'train/*.parquet', 'validation/*.parquet',
                          'calibration/*.parquet', 'test/*.parquet', 'ood/*.parquet'])
DATASET = Path(DATASET)
if RESUME_FROM is not None and not OUTPUT.exists():
    shutil.copytree(RESUME_FROM, OUTPUT)
manifest = json.loads((DATASET / 'manifest.json').read_text())
assert manifest['format'] == 'minifield.magicbox/1.0' and manifest['complete'] is True
assert manifest['mode'] == 'full'
if (manifest['tokenizer']['source_limit'] > SOURCE_TOKENS
        or manifest['tokenizer']['schema_limit'] > SCHEMA_TOKENS):
    raise ValueError('Dataset token limits exceed the configured fixed training shape')
print('Fixed shape:', SOURCE_TOKENS, 'source tokens,', SCHEMA_TOKENS,
      'tokens per schema row,', SCHEMA_ROWS, 'schema rows per request')
by_split = {}
for shard in manifest['shards']:
    by_split[shard['split']] = by_split.get(shard['split'], 0) + shard['rows']
print('Dataset:', DATASET, '\nRecords:', by_split)
print('Logical updates per epoch:', (by_split['train'] + REQUESTS * MICROBATCHES - 1) // (REQUESTS * MICROBATCHES))
snapshot_download(encoder.SOURCE.model_id, revision=encoder.SOURCE.revision,
                  allow_patterns=['config.json', 'tokenizer.json', 'model.safetensors'],
                  local_dir=str(MODEL))

## Import reusable training components


In [ ]:
import dataclasses
import hashlib
import json
from pathlib import Path

import jax
import numpy as np

from examples.magicbox import bundle as magicbox_bundle
from examples.magicbox import composition as magicbox
from examples.magicbox import data
from examples.magicbox import source
from minifield_training.batching import schema_fields as batching
from minifield_training.checkpoints import discovery
from minifield_training.checkpoints import training_state
from minifield_training.core import json_io
from minifield_training.engine import training_run
from minifield_training.evaluation import schema_fields as evaluate
from minifield_training.models.lfm2_5 import encoder
from minifield_training.models.magicbox import model
from minifield_training.objectives import schema_fields as objective
from minifield_training.optimizers import adamw
from minifield_training.strategies import pretrained
from minifield_training.strategies import schema_fields as strategy

from minifield_training.supervisor import diagnostics

## Verify data and build the training plan


In [ ]:
devices = training_run.require_devices(DEVICES, 'tpu')
if (
    REQUESTS % DEVICES
    or EPOCHS < 1
    or VALIDATION_RECORDS < 1
    or FINAL_RECORDS < 0
):
    raise ValueError(
        "Invalid request topology, epoch count, or evaluation bound"
    )
mesh = (
    jax.sharding.Mesh(np.asarray(devices), ("data",))
    if DEVICES > 1
    else None
)
corpus = source.Corpus(
    DATASET, CACHE, allow_sample=False
)
config_path = MODEL / "config.json"
if json_io.digest_file(config_path) != encoder.SOURCE.config_sha256:
    raise ValueError("Pretrained config changed")
cfg = encoder.Adapter().parse_config(
    json_io.object_map(json.loads(config_path.read_text()))
)
fusion = model.Config(
    encoder_width=cfg.hidden_size, row_chunk=ROW_CHUNK
)
if SCHEMA_SEQUENCES is None:
    # Evaluation packs with the training shape, so every split must fit.
    SCHEMA_SEQUENCES = corpus.packed_sequences(
        sorted({str(shard["split"]) for shard in corpus.shards}), SCHEMA_TOKENS
    )
print('Packed schema encoder rows per request:', SCHEMA_SEQUENCES,
      f'({SCHEMA_SEQUENCES * SCHEMA_TOKENS} tokens; unpacked would be {SCHEMA_ROWS * SCHEMA_TOKENS})')
shape = batching.Shape(
    MICROBATCHES,
    REQUESTS,
    SOURCE_TOKENS,
    SCHEMA_TOKENS,
    SCHEMA_ROWS,
    cfg.vocab_size,
    0,
    schema_sequences=SCHEMA_SEQUENCES,
)
if (
    max(shape.source_tokens, shape.schema_tokens)
    > encoder.MAX_SEQUENCE_LENGTH
):
    raise ValueError("Requested tokens exceed the encoder context")
batches = batching.SchemaBatchStrategy(
    shape, objective.balance_types, fixed_shape=True
)
stream = source.training_stream(corpus, batches, SEED, EPOCHS)
inventory = magicbox.inventory(cfg, fusion)
print('Trainable parameters:', inventory.trainable_parameter_count, 'Frozen:', inventory.frozen_names)
optimizer = adamw.AdamWConfig(learning_rate=LEARNING_RATE)
identity = {
    "source": dataclasses.asdict(encoder.SOURCE),
    "encoder": dataclasses.asdict(cfg),
    "fusion": dataclasses.asdict(fusion),
    "shape": {
        key: value
        for key, value in dataclasses.asdict(shape).items()
        if key not in ("vocab_size", "pad_token_id")
    },
    "seed": SEED,
    "bf16": True,
    "devices": DEVICES,
    "contract": data.FORMAT,
    "batching": "fixed-shape-packed-schema/1",
    "implementation": "magicbox-jax/2-frozen-token-embeddings",
}
source_id = hashlib.sha256(json_io.canonical(identity).encode()).hexdigest()
checkpoints = OUTPUT / "checkpoints"
resume = discovery.latest_checkpoint(
    checkpoints,
    run_id=RUN_ID,
    data_sha256=corpus.identity,
    source_id=source_id,
    reject_mismatched=True,
)

print('Resume:', resume, 'Updates per epoch:', stream.updates_per_epoch)
print(diagnostics.memory_snapshot())

## Load pretrained weights


In [ ]:
if resume is None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        _, parameters = pretrained.load_verified(MODEL, encoder.SOURCE, encoder.Adapter())
        parameters.update(model.initialize(fusion, jax.random.PRNGKey(SEED)))
    print('Loaded parameters:', sum(value.size for value in parameters.values()))
else:
    print('Weights will be restored with optimizer state in the next cell:', resume)

## Initialize or restore optimizer state


In [ ]:
if resume:
    current, cursor = training_state.load(
        resume,
        inventory,
        optimizer_id=optimizer.implementation_identity,
        run_id=RUN_ID,
        data_sha256=corpus.identity,
        source_id=source_id,
    )
else:
    current = adamw.initialize_state(parameters, inventory)
    del parameters
    cursor = training_state.Cursor(
        RUN_ID, corpus.identity, source_id, 0
    )
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "run.json").write_text(
    json.dumps(
        {
            **identity,
            "optimizer": dataclasses.asdict(optimizer),
            "data_sha256": corpus.identity,
            "source_id": source_id,
            "updates_per_epoch": stream.updates_per_epoch,
        },
        indent=2,
    )
)

print('Next update:', cursor.next_batch, diagnostics.memory_snapshot())

## Place state on TPU and inspect the first batch


In [ ]:
evaluator = evaluate.Evaluator(
    corpus.records,
    evaluate.Predictor(
        magicbox.bind(cfg, fusion, bf16=True), batches
    ),
    names=data.KINDS,
)

update = strategy.make_step(
    magicbox.bind(cfg, fusion, training=True, bf16=True), inventory, optimizer, mesh=mesh,
)
placement = (jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec())
             if mesh is not None else jax.sharding.SingleDeviceSharding(devices[0]))
current = jax.device_put(current, placement)
jax.block_until_ready(current)
first_batches = stream(cursor.next_batch, None)
try:
    first_update = next(first_batches) if cursor.next_batch < EPOCHS * stream.updates_per_epoch else None
finally:
    first_batches.close()
physical_batch = ({key: value[0] for key, value in first_update.microbatches.items()}
                  if first_update is not None else None)
print('First physical batch:', {key: (value.shape, str(value.dtype))
                                for key, value in (physical_batch or {}).items()})
print(diagnostics.memory_snapshot())

## Lower the complete training step


In [ ]:
if first_update is not None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        print('Lowering the training step to compiler IR', flush=True)
        lowered_step = update.lower(current, first_update.microbatches, first_update.active)
    print('Training step lowering finished', flush=True)

## Compile the complete training step


In [ ]:
if first_update is not None:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        print('Compiling the training step executable', flush=True)
        compiled_step = lowered_step.compile()
    print('Training step compilation finished', flush=True)
    print('Device memory:', compiled_step.memory_analysis(), flush=True)

## Execute the first 2 optimizer updates and save a checkpoint


In [ ]:
def report(event):
    line = json.dumps(event)
    print(line, flush=True)
    with (OUTPUT / 'progress.jsonl').open('a', encoding='utf-8') as log:
        log.write(line + '\n')

preflight_target = min(2, SMOKE_STEPS) if RUN_MODE == 'smoke' else 2
preflight_remaining = min(preflight_target, EPOCHS * stream.updates_per_epoch) - cursor.next_batch
if preflight_remaining > 0:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        current, cursor = training_run.run(
            None, current, update, inventory,
            training_run.RunConfig(SEED, 2, 1, max_steps=preflight_remaining, max_seconds=3600),
            checkpoint_root=checkpoints, optimizer_id=optimizer.implementation_identity,
            cursor=cursor, report=report, required_platform='tpu', batch_source=stream,
            strict_compiles=True,
        )
print('Committed updates:', cursor.next_batch)
if cursor.next_batch > 0:
    training_state.verify_roundtrip(
        checkpoints / f'step-{cursor.next_batch:08d}', current, cursor, inventory,
        optimizer.implementation_identity,
    )
    print('Checkpoint parameters, moments, step, and cursor verified', flush=True)


## Validate the startup checkpoint


In [ ]:
with diagnostics.monitor(OUTPUT / 'diagnostics'):
    metrics = evaluator.run(current['params'], 'validation', 8)
print(json.dumps(metrics, indent=2))

## Continue the selected smoke or full run


In [ ]:
target = min(SMOKE_STEPS, EPOCHS * stream.updates_per_epoch) if RUN_MODE == 'smoke' else EPOCHS * stream.updates_per_epoch
remaining = target - cursor.next_batch
if remaining > 0:
    with diagnostics.monitor(OUTPUT / 'diagnostics'):
        current, cursor = training_run.run(
            None, current, update, inventory,
            training_run.RunConfig(SEED, 250, 10, max_steps=remaining, max_seconds=SESSION_HOURS * 3600),
            checkpoint_root=checkpoints, optimizer_id=optimizer.implementation_identity,
            cursor=cursor, evaluate=evaluator.callback(OUTPUT / 'metrics', VALIDATION_RECORDS),
            report=report, required_platform='tpu', batch_source=stream,
            strict_compiles=True,
        )
print('Committed updates:', cursor.next_batch)

## Evaluate and export the trained bundle


In [ ]:
completed = cursor.next_batch >= EPOCHS * stream.updates_per_epoch
if completed or RUN_MODE == 'smoke':
    for split in ("validation", "calibration", "test", "ood"):
        if any(shard["split"] == split for shard in corpus.shards):
            metrics = evaluator.run(
                current["params"], split, FINAL_RECORDS
            )
            (OUTPUT / f"final-{split}.json").write_text(
                json.dumps(metrics, indent=2)
            )
            print(json.dumps({"split": split, **metrics}), flush=True)
bundle = OUTPUT / f"bundle-{cursor.next_batch:08d}"
if not bundle.exists():
    magicbox_bundle.save(
        bundle,
        current["params"],
        cfg,
        fusion,
        encoder_config=config_path,
        tokenizer=DATASET / "tokenizer",
        step=cursor.next_batch,
    )
print(
    json.dumps(
        {
            "bundle": str(bundle),
            "completed_epochs": completed,
            "next_batch": cursor.next_batch,
        }
    ),
    flush=True,
)

## Reload the exported bundle and predict


In [ ]:
from examples.magicbox import tokenizer

loaded_cfg, loaded_fusion, loaded_parameters, decode_config = magicbox_bundle.load(bundle)
adapter = tokenizer.Adapter(bundle / 'tokenizer')
request = {
    'state': 'Ada is 37 years old and lives in London.',
    'questions': {'name': {'type': 'extract', 'instructions': 'Person name'}},
}
record = data.compile_record('inference', request, {}, adapter.encode)
predictor = evaluate.Predictor(
    magicbox.bind(loaded_cfg, loaded_fusion, bf16=True), batches,
    presence_threshold=float(str(decode_config['presence_threshold'])),
)
predictions, _ = predictor.score(loaded_parameters, record, include_losses=False)
print(json.dumps(data.format_results(record, predictions), indent=2))